# Phase 4: LLMs & Prompt Engineering
## Day 18: PromptEngineering

Date: 2026-04-24

### Learning objectives
- Write clear zero-shot prompts.
- Use few-shot examples to guide model behavior.
- Ask for brief reasoning without exposing hidden chain-of-thought.
- Use role prompting for better context.
- Avoid common prompt mistakes.

In [ ]:
import json
import re
import textwrap
from pprint import pprint

def show(title, content):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    print(textwrap.dedent(str(content)).strip())

print("Setup complete. We will use small mock functions so every example runs locally.")

In [ ]:
campaign_summaries = [
    {
        "campaign": "Spring Coffee Push",
        "channel": "Instagram",
        "spend_eur": 1200,
        "clicks": 3420,
        "conversions": 184,
        "notes": "Strong CTR. Conversions improved after adding a limited-time discount."
    },
    {
        "campaign": "Bank App Onboarding",
        "channel": "Email",
        "spend_eur": 800,
        "clicks": 980,
        "conversions": 42,
        "notes": "Low conversion. Email subject line may be too generic."
    },
    {
        "campaign": "Yoga Studio Trial",
        "channel": "TikTok",
        "spend_eur": 650,
        "clicks": 2100,
        "conversions": 165,
        "notes": "Short videos performed well. Audience responded to beginner-friendly copy."
    },
]

support_messages = [
    "I cannot log into my account after resetting my password.",
    "The delivery was late but the support agent was very helpful.",
    "Please cancel my subscription before the next billing cycle.",
]

pprint(campaign_summaries[0])
print("\nSample support message:", support_messages[0])

## 1. Prompt anatomy

A prompt is not just a question. It usually has a task, context, constraints, and an output format.

Good prompts make the model's job obvious. They reduce guessing.

In [ ]:
def build_prompt(task, context, constraints=None, output_format=None):
    constraints = constraints or []
    prompt = f"Task:\n{task}\n\nContext:\n{context}"
    if constraints:
        prompt += "\n\nConstraints:\n" + "\n".join(f"- {c}" for c in constraints)
    if output_format:
        prompt += f"\n\nOutput format:\n{output_format}"
    return prompt

prompt = build_prompt(
    task="Summarize the campaign performance for a marketing manager.",
    context=json.dumps(campaign_summaries[0], indent=2),
    constraints=["Use 2 short bullets", "Mention one risk"],
    output_format="Bullet list"
)

show("A clear prompt structure", prompt)

## 2. Zero-shot prompting

Zero-shot means you ask the model to do a task without examples.

This works best for simple and common tasks, such as classification, summarization, rewriting, and extraction.

In [ ]:
def mock_zero_shot_classifier(message):
    text = message.lower()
    if "cancel" in text or "subscription" in text:
        return "billing_or_cancellation"
    if "log" in text or "password" in text:
        return "login_problem"
    if "late" in text or "delivery" in text:
        return "delivery_experience"
    return "other"

zero_shot_prompt = build_prompt(
    task="Classify this customer message into one label.",
    context=support_messages[0],
    constraints=["Use only one label", "Do not explain"],
    output_format="login_problem | billing_or_cancellation | delivery_experience | other"
)

show("Zero-shot prompt", zero_shot_prompt)
print("\nMock model output:", mock_zero_shot_classifier(support_messages[0]))

In [ ]:
for msg in support_messages:
    print(f"Message: {msg}")
    print(f"Label:   {mock_zero_shot_classifier(msg)}")
    print()

## 3. Few-shot prompting

Few-shot means you include examples before asking for the real answer.

This is useful when you want a specific style, label set, or output structure.

In [ ]:
few_shot_prompt = '''
Task:
Classify the message into one label.

Labels:
- login_problem
- billing_or_cancellation
- delivery_experience
- other

Examples:
Message: "I forgot my password and cannot access the app."
Label: login_problem

Message: "I want to stop my plan today."
Label: billing_or_cancellation

Message: "My order arrived 40 minutes late."
Label: delivery_experience

Now classify:
Message: "Please cancel my subscription before the next billing cycle."
Label:
'''

show("Few-shot prompt", few_shot_prompt)
print("\nMock model output:", mock_zero_shot_classifier("Please cancel my subscription before the next billing cycle."))

In [ ]:
def make_few_shot_prompt(examples, new_message):
    lines = [
        "Task:",
        "Classify the message into one label.",
        "",
        "Examples:"
    ]
    for message, label in examples:
        lines.append(f'Message: "{message}"')
        lines.append(f"Label: {label}")
        lines.append("")
    lines.append("Now classify:")
    lines.append(f'Message: "{new_message}"')
    lines.append("Label:")
    return "\n".join(lines)

examples = [
    ("I forgot my password and cannot access the app.", "login_problem"),
    ("I want to stop my plan today.", "billing_or_cancellation"),
    ("My order arrived 40 minutes late.", "delivery_experience"),
]

prompt = make_few_shot_prompt(examples, support_messages[1])
show("Generated few-shot prompt", prompt)
print("\nMock model output:", mock_zero_shot_classifier(support_messages[1]))

## 4. Reasoning prompts without hidden chain-of-thought

For complex tasks, ask the model to reason. But do not ask it to reveal hidden chain-of-thought.

A safer pattern is: ask for a short rationale, key checks, or final answer with brief reasoning.

In [ ]:
def campaign_conversion_rate(campaign):
    return campaign["conversions"] / campaign["clicks"]

def brief_campaign_decision(campaign):
    rate = campaign_conversion_rate(campaign)
    if rate >= 0.07:
        decision = "scale"
        reason = "conversion rate is strong"
    elif rate >= 0.04:
        decision = "monitor"
        reason = "conversion rate is acceptable but not excellent"
    else:
        decision = "pause_or_fix"
        reason = "conversion rate is weak"
    return {
        "campaign": campaign["campaign"],
        "conversion_rate": round(rate, 4),
        "decision": decision,
        "brief_rationale": reason
    }

reasoning_prompt = build_prompt(
    task="Decide whether to scale, monitor, or pause this campaign.",
    context=json.dumps(campaign_summaries[0], indent=2),
    constraints=[
        "Think through the key metric internally",
        "Return only the final decision and a brief rationale",
        "Do not include hidden chain-of-thought"
    ],
    output_format='{"decision": "...", "brief_rationale": "..."}'
)

show("Reasoning prompt pattern", reasoning_prompt)
print("\nMock model output:")
pprint(brief_campaign_decision(campaign_summaries[0]))

In [ ]:
for campaign in campaign_summaries:
    result = brief_campaign_decision(campaign)
    print(f"{result['campaign']}: {result['decision']} because {result['brief_rationale']} "
          f"(CVR={result['conversion_rate']})")

## 5. Role prompting

Role prompting gives the model a useful perspective.

Use it when the answer should follow a specific professional style, such as data scientist, product manager, teacher, or customer support agent.

In [ ]:
role_prompt = '''
System role:
You are a senior data scientist explaining campaign results to a non-technical marketing manager.

User task:
Explain whether this campaign looks healthy.

Campaign:
{
  "campaign": "Yoga Studio Trial",
  "channel": "TikTok",
  "spend_eur": 650,
  "clicks": 2100,
  "conversions": 165,
  "notes": "Short videos performed well. Audience responded to beginner-friendly copy."
}

Output:
Use 3 simple bullets. Avoid technical jargon.
'''

show("Role prompting example", role_prompt)

result = brief_campaign_decision(campaign_summaries[2])
print("\nMock model-style answer:")
print(f"- The campaign looks healthy.")
print(f"- It has a strong conversion rate of {result['conversion_rate']:.1%}.")
print("- The beginner-friendly TikTok videos seem worth testing further.")

In [ ]:
def create_role_prompt(role, task, data, style):
    return f'''
System role:
You are {role}.

User task:
{task}

Data:
{json.dumps(data, indent=2)}

Style:
{style}
'''.strip()

prompt = create_role_prompt(
    role="a careful AI engineer",
    task="Review whether this summary is enough for structured extraction.",
    data=campaign_summaries[1],
    style="Give 2 bullets and one missing-data warning."
)

show("Reusable role prompt", prompt)

## 6. Output format control

Models follow instructions better when the output format is explicit.

For data pipelines, prefer JSON schemas over vague text answers.

In [ ]:
json_prompt = build_prompt(
    task="Extract campaign fields from the summary.",
    context="Campaign Spring Coffee Push spent 1200 EUR, got 3420 clicks, and 184 conversions on Instagram.",
    constraints=["Return valid JSON only", "Use null for missing values"],
    output_format='''
{
  "campaign": string,
  "channel": string | null,
  "spend_eur": number | null,
  "clicks": integer | null,
  "conversions": integer | null
}
'''
)

show("JSON output prompt", json_prompt)

mock_json_output = {
    "campaign": "Spring Coffee Push",
    "channel": "Instagram",
    "spend_eur": 1200,
    "clicks": 3420,
    "conversions": 184
}

print("\nMock model output:")
print(json.dumps(mock_json_output, indent=2))

In [ ]:
def parse_mock_campaign_summary(text):
    campaign = re.search(r"Campaign ([\w\s]+?) spent", text)
    spend = re.search(r"spent (\d+) EUR", text)
    clicks = re.search(r"got (\d+) clicks", text)
    conversions = re.search(r"and (\d+) conversions", text)
    channel = re.search(r"on ([A-Za-z]+)", text)
    return {
        "campaign": campaign.group(1) if campaign else None,
        "channel": channel.group(1) if channel else None,
        "spend_eur": int(spend.group(1)) if spend else None,
        "clicks": int(clicks.group(1)) if clicks else None,
        "conversions": int(conversions.group(1)) if conversions else None,
    }

summary_text = "Campaign Spring Coffee Push spent 1200 EUR, got 3420 clicks, and 184 conversions on Instagram."
parsed = parse_mock_campaign_summary(summary_text)
pprint(parsed)

## 7. Tricky bits

Prompt problems are often simple: vague tasks, missing output format, too many tasks at once, or contradictory instructions.

The code below shows common prompt checks.

In [ ]:
bad_prompts = [
    "Analyze this.",
    "Give me a short answer with full detailed reasoning and all possible edge cases.",
    "Extract the data and maybe summarize it nicely.",
    "Return JSON but also explain the result in a paragraph."
]

def check_prompt(prompt):
    warnings = []
    if len(prompt.split()) < 6:
        warnings.append("Too vague. Add task, context, and output format.")
    if "json" in prompt.lower() and ("explain" in prompt.lower() or "paragraph" in prompt.lower()):
        warnings.append("Mixed output format. JSON-only prompts should not ask for prose.")
    if "short" in prompt.lower() and ("full detailed" in prompt.lower() or "all possible" in prompt.lower()):
        warnings.append("Contradictory length instructions.")
    if "maybe" in prompt.lower():
        warnings.append("Unclear instruction. Replace 'maybe' with a clear requirement.")
    return warnings or ["Looks okay."]

for p in bad_prompts:
    print(f"Prompt: {p}")
    print("Warnings:", check_prompt(p))
    print()

In [ ]:
def require_json_only(prompt):
    if "json" not in prompt.lower():
        raise ValueError("Prompt does not request JSON.")
    if "paragraph" in prompt.lower() or "explain" in prompt.lower():
        raise ValueError("Prompt mixes JSON with prose.")
    return "Prompt is safe for JSON extraction."

test_prompts = [
    "Extract the fields. Return valid JSON only.",
    "Return JSON and explain it in a paragraph."
]

for p in test_prompts:
    try:
        print(require_json_only(p))
    except ValueError as error:
        print("Broken prompt:", error)

## Trick questions

1. Is zero-shot always worse than few-shot?

<details>
<summary>Answer</summary>

No. Zero-shot can be better when the task is simple and examples would add noise.

</details>

2. Should you ask the model to reveal its full chain-of-thought?

<details>
<summary>Answer</summary>

No. Ask for a short rationale, key checks, or final answer with brief reasoning instead.

</details>

3. Why is "Return JSON and explain it" risky?

<details>
<summary>Answer</summary>

It mixes two output formats. This can break JSON parsing.

</details>

4. What is the main purpose of role prompting?

<details>
<summary>Answer</summary>

It gives the model a useful perspective, tone, and decision frame.

</details>

5. When are few-shot examples useful?

<details>
<summary>Answer</summary>

When the task needs a specific label set, style, format, or edge-case behavior.

</details>

## Exercises

Fill in each `___`. Run the cell to check your answer.

In [ ]:
# Exercise 1
# Create a zero-shot prompt that classifies a message into one label.

message = "My order was late and the food was cold."

zero_shot_prompt = ___

assert isinstance(zero_shot_prompt, str)
assert "classify" in zero_shot_prompt.lower()
assert "label" in zero_shot_prompt.lower()
assert message in zero_shot_prompt
print("Exercise 1 passed.")

In [ ]:
# Exercise 2
# Create a few-shot example list with 2 examples.

examples = ___

assert isinstance(examples, list)
assert len(examples) == 2
assert all(isinstance(item, tuple) and len(item) == 2 for item in examples)
print("Exercise 2 passed.")

In [ ]:
# Exercise 3
# Build a role prompt for a data scientist explaining a campaign.

role = ___
task = ___

prompt = create_role_prompt(
    role=role,
    task=task,
    data=campaign_summaries[0],
    style="Use 2 simple bullets."
)

assert "data scientist" in prompt.lower()
assert "Campaign" in prompt or "campaign" in prompt
print("Exercise 3 passed.")

In [ ]:
# Exercise 4
# Write a JSON-only output format instruction.

output_instruction = ___

assert isinstance(output_instruction, str)
assert "json" in output_instruction.lower()
assert "only" in output_instruction.lower()
print("Exercise 4 passed.")

In [ ]:
# Exercise 5
# Choose the better reasoning instruction.

bad_instruction = "Show your full private chain-of-thought."
good_instruction = ___

assert isinstance(good_instruction, str)
assert "brief" in good_instruction.lower() or "rationale" in good_instruction.lower()
assert "chain-of-thought" not in good_instruction.lower()
print("Exercise 5 passed.")

In [ ]:
# Exercise 6
# Repair a vague prompt.

vague_prompt = "Analyze this."
repaired_prompt = ___

assert isinstance(repaired_prompt, str)
assert len(repaired_prompt.split()) >= 12
assert "output" in repaired_prompt.lower() or "format" in repaired_prompt.lower()
print("Exercise 6 passed.")

In [ ]:
# Exercise 7
# Create a prompt using the build_prompt function.

my_prompt = build_prompt(
    task=___,
    context=___,
    constraints=[___, ___],
    output_format=___
)

assert "Task:" in my_prompt
assert "Context:" in my_prompt
assert "Constraints:" in my_prompt
assert "Output format:" in my_prompt
print("Exercise 7 passed.")

## Solutions

<details>
<summary>Exercise 1 solution</summary>

```python
zero_shot_prompt = '''
Classify this customer message into one label.

Message:
My order was late and the food was cold.

Labels:
delivery_experience, billing_or_cancellation, login_problem, other

Return only the label.
'''
```

</details>

<details>
<summary>Exercise 2 solution</summary>

```python
examples = [
    ("I forgot my password.", "login_problem"),
    ("Please cancel my plan.", "billing_or_cancellation"),
]
```

</details>

<details>
<summary>Exercise 3 solution</summary>

```python
role = "a data scientist"
task = "Explain whether this campaign looks healthy."
```

</details>

<details>
<summary>Exercise 4 solution</summary>

```python
output_instruction = "Return valid JSON only."
```

</details>

<details>
<summary>Exercise 5 solution</summary>

```python
good_instruction = "Give the final answer with a brief rationale."
```

</details>

<details>
<summary>Exercise 6 solution</summary>

```python
repaired_prompt = "Analyze this campaign summary. Return 3 bullets covering performance, risk, and next action. Use simple language."
```

</details>

<details>
<summary>Exercise 7 solution</summary>

```python
my_prompt = build_prompt(
    task="Summarize this campaign for a marketing manager.",
    context=json.dumps(campaign_summaries[0]),
    constraints=["Use 2 bullets", "Mention one risk"],
    output_format="Bullet list"
)
```

</details>

## Cumulative review exercises

These mix topics from Days 8 to 17. Fill in `___` and run each cell.

In [ ]:
# Review 1: Ensembles
# Choose the ensemble method that combines many decision trees using bagging.

ensemble_method = ___

assert ensemble_method.lower() == "random forest"
print("Review 1 passed.")

In [ ]:
# Review 2: Classification metrics
# Calculate precision from true positives and false positives.

tp = 80
fp = 20

precision = ___

assert abs(precision - 0.8) < 1e-9
print("Review 2 passed.")

In [ ]:
# Review 3: SHAP idea
# Complete the sentence.

shap_sentence = ___

assert "feature" in shap_sentence.lower()
assert "contribution" in shap_sentence.lower() or "impact" in shap_sentence.lower()
print("Review 3 passed.")

In [ ]:
# Review 4: TF-IDF
# Pick the term that should usually get higher IDF.

common_word = "the"
rare_word = "chargeback"

higher_idf_word = ___

assert higher_idf_word == rare_word
print("Review 4 passed.")

In [ ]:
# Review 5: Embeddings
# A cosine similarity close to 1 usually means the vectors are very similar.

cosine_similarity = ___

assert 0.9 <= cosine_similarity <= 1.0
print("Review 5 passed.")

In [ ]:
# Review 6: Hugging Face
# Name the common function used for quick pretrained tasks.

hf_function = ___

assert hf_function == "pipeline"
print("Review 6 passed.")

In [ ]:
# Review 7: Fine-tuning BERT
# Pick the object that usually stores epochs, batch size, and learning rate in Trainer API.

trainer_config_object = ___

assert trainer_config_object == "TrainingArguments"
print("Review 7 passed.")

In [ ]:
# Review 8: Complaint classification
# Create a simple label mapping.

label_to_id = ___

assert isinstance(label_to_id, dict)
assert all(isinstance(v, int) for v in label_to_id.values())
assert len(label_to_id) >= 2
print("Review 8 passed.")

In [ ]:
# Review 9: OpenAI API roles
# Fill the standard chat roles.

roles = ___

assert roles == ["system", "user", "assistant"]
print("Review 9 passed.")

In [ ]:
# Review 10: Ollama REST API
# Fill the usual local Ollama generate endpoint.

ollama_url = ___

assert ollama_url == "http://localhost:11434/api/generate"
print("Review 10 passed.")

## Cumulative review solutions

<details>
<summary>Show solutions</summary>

```python
# Review 1
ensemble_method = "random forest"

# Review 2
precision = tp / (tp + fp)

# Review 3
shap_sentence = "SHAP explains each feature contribution to a model prediction."

# Review 4
higher_idf_word = rare_word

# Review 5
cosine_similarity = 0.95

# Review 6
hf_function = "pipeline"

# Review 7
trainer_config_object = "TrainingArguments"

# Review 8
label_to_id = {"billing": 0, "technical": 1, "delivery": 2}

# Review 9
roles = ["system", "user", "assistant"]

# Review 10
ollama_url = "http://localhost:11434/api/generate"
```

</details>

In [ ]:
cheat_sheet = '''
DAY 18 CHEAT SHEET: PROMPT ENGINEERING

Prompt anatomy:
- Task: what the model should do
- Context: the data or background
- Constraints: rules the model must follow
- Output format: shape of the answer

Zero-shot:
- No examples
- Best for simple tasks
- Example: "Classify this message into one label"

Few-shot:
- Add examples before the real task
- Best for style, labels, format, and edge cases

Reasoning:
- Ask for a brief rationale or key checks
- Do not ask for hidden chain-of-thought

Role prompting:
- Give the model a useful perspective
- Example: "You are a senior data scientist"

Common pitfalls:
- Vague task
- Missing output format
- Contradictory instructions
- Mixing JSON with prose
- Too many tasks in one prompt
'''

print(cheat_sheet)

## Next up: Day 19 — StructuredOutputExtraction

You will learn how to force JSON output, validate it with Pydantic, and repair broken responses.